# Import

In [1]:
import numpy as np
import json
from scipy.sparse import load_npz,save_npz,diags,csr_matrix
import scipy.sparse as sp
import pandas as pd
import os
import requests
from io import BytesIO
from tqdm import tqdm
from scipy.sparse.linalg import eigsh
from scipy.spatial.distance import pdist, squareform
import matplotlib.pyplot as plt
from pathlib import Path
from matplotlib.backends.backend_pdf import PdfPages
from pypdf import PdfReader, PdfWriter
from tempfile import NamedTemporaryFile
import networkx as nx
import pickle
import gseapy as gp
import mygene
from IPython.display import display, HTML
import re
from collections import deque
from goatools.obo_parser import GODag
import math
from itertools import combinations
from collections import Counter
from gseapy.parser import read_gmt
import time
import random
import ast

In [2]:
pd.set_option('display.width', None)      # No line-wrapping
pd.set_option('display.max_columns', None)  # Show all columns

# Dependency
* DGIDB_hypergraph
* DDBC

# Prep

## Loading variables

In [3]:
DISEASE = input("Disease: ")
DISEASE_FOLDER = f"../output/{DISEASE}/"
DGIDB_DIRECTORY = f"../../Gen_Hypergraph/output/DGIDB_{DISEASE}/"
MSIGDB_DIRECTORY = "../../Gen_Hypergraph/output/MSigDB_Full/"
RESULT_GRAPH = "result_graph"

with open(DISEASE_FOLDER + "gene_to_index_distinct.json", "r") as file:
    gene_to_index_distinct = json.load(file)
    
try:
    with open(DGIDB_DIRECTORY + f"gene_to_index.json", "r") as file:
        DGIDB_gene_to_index = json.load(file)
except FileNotFoundError:
    DGIDB_gene_to_index = {}
    print("File not found. Setting DGIDB_gene_to_index to be {}.")

In [4]:
## ORIGINAL
index_to_gene_distinct = {v: k for k, v in gene_to_index_distinct.items()}

In [5]:
# Loading result graph and communities
with open(f"{DISEASE_FOLDER}/result_communities_selected.pkl", "rb") as f:
    communities_selected = pickle.load(f)
with open(f"{DISEASE_FOLDER}/result_communities.pkl", "rb") as f:
    communities = pickle.load(f)
with open(f"{DISEASE_FOLDER}/{RESULT_GRAPH}.pkl", "rb") as f:
    graph = pickle.load(f)

In [6]:
for c in communities_selected:
    print(len(c))

909
985
1206
907
931
875
661
546
395
341
300
135


## Helpful functions (big object, drop NAN)

In [7]:
# Helpful functions
def drop_nan_from_communities(communities):
    cleaned_communities = []
    total_dropped = 0

    for i, community in enumerate(communities):
        cleaned = []
        dropped = 0
        for g in community:
            if g is None or (isinstance(g, float) and math.isnan(g)):
                dropped += 1
            else:
                cleaned.append(g)
        cleaned_communities.append(cleaned)
        total_dropped += dropped
        print(f"Community {i}: dropped {dropped} NaN entries")

    print(f"\nTotal dropped across all communities: {total_dropped}")
    return cleaned_communities

def big_objects(n=10, min_mb=1):
    """
    Show the largest objects currently in memory.
    
    Parameters
    ----------
    n : int
        Number of top objects to show.
    min_mb : float
        Minimum size (in MB) to include.
    """
    import sys
    import numpy as np
    import pandas as pd
    import scipy.sparse as sp
    from IPython import get_ipython

    def get_size(obj):
        try:
            if isinstance(obj, np.ndarray):
                return obj.nbytes
            elif isinstance(obj, pd.DataFrame) or isinstance(obj, pd.Series):
                return obj.memory_usage(deep=True).sum()
            elif sp.issparse(obj):
                return (obj.data.nbytes +
                        obj.indptr.nbytes +
                        obj.indices.nbytes)
            else:
                return sys.getsizeof(obj)
        except Exception:
            return 0

    ip = get_ipython()
    if ip is None:
        ns = globals()
    else:
        ns = ip.user_ns

    items = []
    for name, val in ns.items():
        if name.startswith('_'):
            continue  # skip internals
        size = get_size(val)
        if size > min_mb * 1024 ** 2:
            items.append((name, type(val).__name__, size))

    items.sort(key=lambda x: x[2], reverse=True)

    print(f"{'Variable':30s} {'Type':25s} {'Size (MB)':>10s}")
    print("-" * 70)
    for name, t, size in items[:n]:
        print(f"{name:30s} {t:25s} {size / 1024 ** 2:10.2f}")

## Index to NCBI

In [8]:
# Convert index to ncbi
def index_to_ncbi(comms,index_to_ncbi_dict = index_to_gene_distinct):
    comms_ncbi = [list(map(index_to_ncbi_dict.get, c)) for c in comms]
    return comms_ncbi

In [9]:
communities_ncbi = index_to_ncbi(communities_selected,index_to_gene_distinct)
print(communities_ncbi)
print(len(communities_ncbi))
with open(f"{DISEASE_FOLDER}/result_communities_ncbi_selected.pkl", "wb") as f:
    pickle.dump(communities_ncbi, f)

[['55884', '57222', '10724', '157567', '55858', '9019', '54978', '9980', '4820', '57720', '55186', '84272', '113419', '11163', '9991', '64327', '7163', '55754', '9522', '91404', '387263', '6397', '9747', '6651', '10807', '169200', '9202', '8763', '50717', '56987', '4774', '754', '5954', '4154', '23635', '23673', '64423', '124565', '10522', '55608', '6666', '64776', '140890', '3338', '10180', '65084', '54765', '440026', '10314', '26268', '57700', '51290', '57798', '120', '50999', '126321', '8543', '104472715', '9584', '3995', '23176', '55005', '23151', '23200', '51663', '55325', '5813', '51088', '57460', '54788', '162427', '9145', '23484', '64746', '7873', '54629', '7267', '55900', '23199', '23613', '23157', '7095', '55556', '54708', '81627', '55317', '9325', '11079', '136319', '51108', '25957', '127262', '91966', '88455', '23215', '1429', '81572', '9910', '55905', '91304', '8073', '79627', '6509', '23029', '221035', '58497', '80212', '84939', '27230', '26225', '23065', '79573', '51016'

In [10]:
communities_ncbi_full = index_to_ncbi(communities,index_to_gene_distinct)
print(communities_ncbi_full)
print(len(communities_ncbi_full))
with open(f"{DISEASE_FOLDER}/result_communities_ncbi.pkl", "wb") as f:
    pickle.dump(communities_ncbi_full, f)

[['28', '56', '100', '118', '120', '132', '141', '143', '162', '164', '166', '175', '178', '205', '226', '250', '267', '271', '286', '287', '310', '353', '372', '373', '377', '378', '381', '400', '402', '421', '427', '439', '440', '444', '473', '475', '520', '523', '547', '550', '576', '577', '586', '631', '636', '662', '665', '676', '734', '738', '750', '754', '755', '757', '770', '811', '819', '821', '829', '830', '832', '833', '889', '987', '989', '1039', '1054', '1080', '1102', '1119', '1120', '1122', '1153', '1158', '1174', '1176', '1181', '1182', '1192', '1198', '1201', '1203', '1209', '1266', '1314', '1315', '1357', '1362', '1371', '1389', '1400', '1406', '1411', '1429', '1506', '1527', '1603', '1654', '1656', '1657', '1731', '1773', '1775', '1777', '1797', '1801', '1819', '1877', '1891', '1939', '1951', '1952', '1983', '1984', '2027', '2039', '2054', '2121', '2135', '2137', '2139', '2171', '2195', '2218', '2235', '2286', '2310', '2314', '2504', '2509', '2519', '2523', '2530', '

## NCBI to HGNC

In [11]:
hgnc = pd.read_csv("../../Data/hgnc_complete_set.txt", sep="\t", dtype=str)
ncbi_to_hgnc_dict = dict(
    zip(
        hgnc["entrez_id"].dropna(),
        hgnc.loc[hgnc["entrez_id"].notna(), "symbol"]
    )
)

def ncbi_to_HGNC(comms_ncbi):
    comms_HGNC = []
    for community in comms_ncbi:
        symbols = [ncbi_to_hgnc_dict.get(n) for n in community]
        comms_HGNC.append(symbols)
    return comms_HGNC

In [12]:
# # NCBI to HGNC symbol
# def ncbi_to_HGNC(comms_ncbi):
#     comms_HGNC = []
#     for community in comms_ncbi:
#         mg = mygene.MyGeneInfo()
#         entrez_ids = [str(e) for e in community]

#         results = mg.querymany(
#             entrez_ids,
#             scopes="entrezgene",
#             fields="symbol",
#             species="human"
#         )

#         # Build a mapping: input ID -> symbol (or None)
#         id_to_symbol = {}
#         for r in results:
#             q = str(r.get("query"))
#             id_to_symbol[q] = r.get("symbol") if not r.get("notfound") else None

#         # Preserve original order
#         symbols = [id_to_symbol.get(str(e), None) for e in entrez_ids]
#         comms_HGNC.append(symbols)
#     return comms_HGNC


In [13]:
COMMUNITIES_HGNC = ncbi_to_HGNC(communities_ncbi)
COMMUNITIES_HGNC_full = ncbi_to_HGNC(communities_ncbi_full)

In [14]:
print(len(COMMUNITIES_HGNC))

12


In [15]:
COMMUNITIES_HGNC = drop_nan_from_communities(COMMUNITIES_HGNC)
COMMUNITIES_HGNC_full = drop_nan_from_communities(COMMUNITIES_HGNC_full)

Community 0: dropped 0 NaN entries
Community 1: dropped 0 NaN entries
Community 2: dropped 1 NaN entries
Community 3: dropped 0 NaN entries
Community 4: dropped 0 NaN entries
Community 5: dropped 4 NaN entries
Community 6: dropped 0 NaN entries
Community 7: dropped 0 NaN entries
Community 8: dropped 1 NaN entries
Community 9: dropped 1 NaN entries
Community 10: dropped 0 NaN entries
Community 11: dropped 1 NaN entries

Total dropped across all communities: 8
Community 0: dropped 4 NaN entries
Community 1: dropped 5 NaN entries
Community 2: dropped 8 NaN entries
Community 3: dropped 2 NaN entries
Community 4: dropped 1 NaN entries
Community 5: dropped 8 NaN entries
Community 6: dropped 2 NaN entries
Community 7: dropped 3 NaN entries
Community 8: dropped 4 NaN entries
Community 9: dropped 1 NaN entries
Community 10: dropped 0 NaN entries
Community 11: dropped 1 NaN entries
Community 12: dropped 0 NaN entries
Community 13: dropped 0 NaN entries
Community 14: dropped 0 NaN entries
Communi

In [16]:
with open(f"{DISEASE_FOLDER}/result_communities_HGNC_selected.pkl", "wb") as f:
    pickle.dump(COMMUNITIES_HGNC, f)
with open(f"{DISEASE_FOLDER}/result_communities_HGNC.pkl", "wb") as f:
    pickle.dump(COMMUNITIES_HGNC_full, f)

In [17]:
print(len(COMMUNITIES_HGNC))
print(len(COMMUNITIES_HGNC_full))

12
19


# Categoization Prep

### GO-slim

In [18]:
DATA_DIRECTORY = "../../data"
GO_OBO = f"{DATA_DIRECTORY}/GO/go-basic.obo"            # put the file in your working dir (or give full path)
GOSLIM_OBO = f"{DATA_DIRECTORY}/GO/goslim_generic.obo"  # swap to another slim if you prefer
GOSLIM_PIR_OBO = f"{DATA_DIRECTORY}/GO/goslim_pir.obo"  # swap to another slim if you prefer
GOSLIM_YEAST_OBO = f"{DATA_DIRECTORY}/GO/goslim_yeast.obo"
GOSLIM_AGR_OBO = f"{DATA_DIRECTORY}/GO/goslim_agr.obo"

In [19]:
# GO library
go = GODag(GO_OBO)

# SLIM libraries
slim = GODag(GOSLIM_OBO)
slim_pir = GODag(GOSLIM_PIR_OBO)
slim_yeast = GODag(GOSLIM_YEAST_OBO)
slim_agr = GODag(GOSLIM_AGR_OBO)

slim_ids = set(slim.keys())
slim_pir_ids = set(slim_pir.keys())
slim_yeast_ids = set(slim_yeast.keys())
slim_agr_ids = set(slim_agr.keys())

../../data/GO/go-basic.obo: fmt(1.2) rel(2025-10-10) 42,666 Terms
../../data/GO/goslim_generic.obo: fmt(1.2) rel(go/2025-10-10/subsets/goslim_generic.owl) 205 Terms
../../data/GO/goslim_pir.obo: fmt(1.2) rel(go/2025-10-10/subsets/goslim_pir.owl) 617 Terms
../../data/GO/goslim_yeast.obo: fmt(1.2) rel(go/2025-10-10/subsets/goslim_yeast.owl) 295 Terms
../../data/GO/goslim_agr.obo: fmt(1.2) rel(go/2025-10-10/subsets/goslim_agr.owl) 94 Terms


In [20]:
GO_RE = re.compile(r"(GO:\d{7})")

def get_goid(term: str):
    if isinstance(term, str):
        m = GO_RE.search(term)
        if m:
            return m.group(1)
    raise RuntimeError("Term not found!!")

def get_go_ancestors(go_id):
    """Return a list of ancestor GO term IDs for the given GO ID using QuickGO."""
    url = f"https://www.ebi.ac.uk/QuickGO/services/ontology/go/terms/{go_id}/ancestors"
    headers = {"Accept": "application/json"}

    r = requests.get(url, headers=headers)
    r.raise_for_status()

    data = r.json()
    results = data.get("results", [])
    if not results:
        return []

    # Ancestors come back as a simple list of GO IDs (strings)
    ancestors = results[0].get("ancestors", [])
    return set(ancestors)


def get_go_ancestors_in_slim(go_id):
    ancestors = get_go_ancestors(go_id)
    return slim_ids & ancestors

In [21]:
def get_go_ancestors_at_depth(go_id, depth, include_relations=("is_a", "part_of")):
    """
    Return the set of GO term IDs that are ancestors of `go_id` and have
    absolute depth == `depth` in the GO DAG.

    Parameters
    ----------
    go_id : str
        Starting GO term (e.g., "GO:0051310").
    depth : int
        Absolute depth in the GO DAG (e.g., 3 means all ancestors at depth=3).
    include_relations : tuple[str]
        Relation types to traverse upward, e.g. ("is_a", "part_of", "regulates", ...).

    Returns
    -------
    set[str]
        Ancestor GO IDs whose term.depth == `depth`. Empty set if none.
    """
    if depth < 0:
        return set()
    if go_id not in go:
        return set()

    # One-hop function honoring relation filter
    def parent_ids(term):
        ids = set()
        if "is_a" in include_relations:
            # GOATOOLS usually puts is_a parents here (and sometimes part_of merged)
            ids.update(p.id for p in term.parents)

        rel = getattr(term, "relationship", {}) or {}
        for r in include_relations:
            # relationship entries are already GO IDs
            ids.update(rel.get(r, []))

        # ensure IDs exist in DAG
        return {pid for pid in ids if pid in go}

    result = set()
    frontier = {go_id}
    visited = {go_id}

    # BFS upwards, but pruning branches that are already above the target depth
    while frontier:
        next_frontier = set()
        for node in frontier:
            for pid in parent_ids(go[node]):
                if pid in visited:
                    continue
                visited.add(pid)
                d = go[pid].depth  # absolute depth in DAG

                if d == depth:
                    # ancestor at the exact target depth
                    result.add(pid)
                elif d > depth:
                    # still "below" target depth (further from root),
                    # its parents might reach the target depth
                    next_frontier.add(pid)
                # if d < depth: this branch has gone above the target,
                # and all further ancestors will have depth <= d, so we can skip
        frontier = next_frontier

    return result


### KEGG

In [22]:
def build_kegg_name_to_id(species="hsa"):
    """Map KEGG pathway name -> 'hsaXXXXX' (species-specific)."""
    lines = requests.get(f"https://rest.kegg.jp/list/pathway/{species}").text.strip().splitlines()
    name_to_id = {}
    for ln in lines:
        pid, raw = ln.split("\t")
        pid = pid.replace("path:", "")  # e.g. hsa03010
        # strip " - Homo sapiens (human)" suffix
        name = re.sub(r"\s*-\s*Homo sapiens.*$", "", raw).strip()
        name_to_id[name.lower()] = pid
    return name_to_id

name_to_id = build_kegg_name_to_id("hsa")

In [23]:
def get_kegg_level2(hsa_id: str) -> str | None:
    """
    Return the KEGG Level 2 category for a pathway like 'hsa03040'.
    Example: get_kegg_level2("hsa03040") -> 'Transcription'
    """
    url = f"http://rest.kegg.jp/get/{hsa_id}"
    try:
        text = requests.get(url, timeout=10).text
    except Exception:
        return None

    for line in text.splitlines():
        if line.startswith("CLASS"):
            # CLASS line looks like: CLASS       Genetic Information Processing; Transcription
            parts = [p.strip() for p in line.split(";", maxsplit=2)]
            if len(parts) >= 2:
                return [parts[1]]
            elif len(parts) == 1:
                return [parts[0].replace("CLASS", "").strip()]
    return []

### Reactome

In [24]:
def build_reactome_level_map(level=1, species="9606"):
    """
    Returns { 'R-HSA-xxxxx': ['CategoryNameAtLevel', ...], ... } for the given species.

    Parameters
    ----------
    level : int, default=1
        1-based depth in the Reactome pathway hierarchy:
          - level=1 → top-level Reactome categories (original behavior)
          - level=2 → second-level ancestors, etc.
        If a node is shallower than `level`, the deepest available ancestor
        is used as a fallback.
    species : str, default="9606"
        Taxonomy ID ("9606") or species name ("Homo sapiens").
    """
    if level < 1:
        raise ValueError("level must be >= 1 (1-based depth)")

    # ensure spaces are encoded if a name is used
    species_path = species.replace(" ", "+")
    url = f"https://reactome.org/ContentService/data/eventsHierarchy/{species_path}"
    print(url)
    r = requests.get(url, headers={"Accept": "application/json"}, timeout=300)
    r.raise_for_status()
    trees = r.json()  # list of trees, one per TopLevelPathway

    mapping = {}

    def walk(node, ancestors):
        """
        node: current node dict
        ancestors: list of ancestor nodes from root to parent of `node`
        """
        # ancestors_chain includes current node at the end
        ancestors_chain = ancestors + [node]

        st_id = node.get("stId")
        if st_id:
            # We want the ancestor at depth `level` (1-based).
            # If the path is shorter than `level`, fall back to the deepest one.
            if len(ancestors_chain) >= level:
                cat_node = ancestors_chain[level - 1]
            else:
                cat_node = ancestors_chain[-1]

            cat_name = cat_node.get("name")
            if cat_name:
                mapping.setdefault(st_id, set()).add(cat_name)

        # Recurse into children
        for child in node.get("children", []):
            walk(child, ancestors_chain)

    # Each tree is a top-level pathway
    for top in trees:
        walk(top, [])

    # sets -> sorted lists
    return {k: sorted(v) for k, v in mapping.items()}

In [25]:
# Specific for Reactome: build level map first
reactome_level1 = build_reactome_level_map(level = 1)

https://reactome.org/ContentService/data/eventsHierarchy/9606


# Run Enrichment Analysis

In [26]:
TERM_SCORE_CAP = 1e-5
PERCENTAGE = 0.1

In [27]:
def enrichment(communities,
               term_score_cap,
               percentage, 
               db,
               term_to_category):
    important_terms = pd.DataFrame(columns=["Community Index","Community Size","Term", "Overlap", "Adjusted P-value","Category"])
    i = 0
    num_nonzero_communities = 0
    
    rows = []
    for community in communities:
        # Gene Ontology enrichment
        enr_go = gp.enrichr(
            gene_list=community,
            gene_sets=db,
            organism='Human',
            outdir=None # don't write to disk
        )
        go_df = enr_go.results
        
        # Filter by overlap percentage and adjusted p-value
        mask =  (go_df["Adjusted P-value"] < term_score_cap) & (go_df["Overlap"].apply(lambda x: int(x.split("/")[0])/int(x.split("/")[1]) > percentage))
        filtered = go_df[mask].copy()
        
        # Categorization from GO-Slim
        filtered["Category"] = filtered["Term"].apply(lambda term: term_to_category(term))

        # Get empty count
        empty_count = (filtered["Category"].apply(len) == 0).sum()
        
        
        # Sort
        filtered['Overlap (value)'] = filtered['Overlap'].apply(lambda x: int(x.split("/")[0])/int(x.split("/")[1]))
        filtered = filtered.sort_values(['Adjusted P-value'], ascending=True)
        
        # compute genes involved in enrichment
        community_set = set(community)
        hit_genes = set(";".join(filtered["Genes"].dropna()).split(";"))

        involved = sorted(community_set & hit_genes)
        not_involved = sorted(community_set - hit_genes)

        rows.append({
            "community": i,
            "n_genes": len(community_set),
            "genes_involved": involved,
            "n_involved": len(involved),
            "n_not_involved": len(not_involved)
        })
        
        # Add results to important terms
        if not filtered.empty:
            # print size of community
            print(f"Size of community: {len(community)}")
            
            # print number of filtered terms
            print(f"Number of filtered terms: {len(filtered)}")
            print(f"Number of unmapped terms: {empty_count}")      
            filtered.loc[:, "Community Index"] = i
            filtered.loc[:, "Community Size"] = len(community)
            important_terms = pd.concat([important_terms, filtered], ignore_index=True)
            display(HTML(filtered[["Community Index",'Term','Overlap','Adjusted P-value',"Category"]].head(10).to_html(max_cols=None)))
            num_nonzero_communities += 1

        i += 1
        
    community_coverage_df = pd.DataFrame(rows)
    
    print(f"{num_nonzero_communities} out of {len(communities)} communities had significant GO terms.")
    return important_terms,community_coverage_df

### GO

In [28]:
# GO Analysis; save terms with small size and high p-value
def go_enrichment(communities,
                  term_score_cap,
                  percentage, 
                  slim_ids = slim_yeast_ids,
                  depth = 1):
    important_terms = pd.DataFrame(columns=["Community Index","Community Size","Term", "Overlap", "Adjusted P-value","Category"])
    i = 0
    num_nonzero_communities = 0
    
    rows = []
    for community in communities:
        # Gene Ontology enrichment
        enr_go = gp.enrichr(
            gene_list=community,
            gene_sets=['GO_Biological_Process_2023',
                    'GO_Molecular_Function_2023',
                    'GO_Cellular_Component_2023'],
            organism='Human',
            outdir=None # don't write to disk
        )
        go_df = enr_go.results
        
        # Filter by overlap percentage and adjusted p-value
        mask =  (go_df["Adjusted P-value"] < term_score_cap) & (go_df["Overlap"].apply(lambda x: int(x.split("/")[0])/int(x.split("/")[1]) > percentage))
        filtered = go_df[mask].copy()
        
        # Categorization from GO-Slim
        filtered["id"] = filtered["Term"].apply(get_goid)
        # filtered["Slim_IDs"] = filtered["GO_ID"].apply(get_go_ancestors_in_slim)
        filtered["Slim_IDs"] = filtered["id"].apply(lambda id: get_go_ancestors_at_depth(id, depth=depth, include_relations=("is_a", "part_of")))
        
        # Get empty count
        empty_count = (filtered["Slim_IDs"].apply(len) == 0).sum()
        
        # Get slim names    
        filtered["Category"] = filtered["Slim_IDs"].apply(lambda ids: [go[i].name for i in ids])
        
        # Sort
        filtered['Overlap (value)'] = filtered['Overlap'].apply(lambda x: int(x.split("/")[0])/int(x.split("/")[1]))
        filtered = filtered.sort_values(['Overlap (value)'], ascending=False)
        
        # compute genes involved in enrichment
        community_set = set(community)
        hit_genes = set(";".join(filtered["Genes"].dropna()).split(";"))

        involved = sorted(community_set & hit_genes)
        not_involved = sorted(community_set - hit_genes)

        rows.append({
            "community": i,
            "n_genes": len(community_set),
            "n_involved": len(involved),
            "n_not_involved": len(not_involved),
        })
        
        # Add results to important terms
        if not filtered.empty:
            # print size of community
            print(f"Size of community: {len(community)}")
            
            # print number of filtered terms
            print(f"Number of filtered terms: {len(filtered)}")
            print(f"Number of unmapped terms: {empty_count}")      
            filtered.loc[:, "Community Index"] = i
            filtered.loc[:, "Community Size"] = len(community)
            important_terms = pd.concat([important_terms, filtered], ignore_index=True)
            display(HTML(filtered[["Community Index",'Term','Overlap','Adjusted P-value',"Slim_IDs","Category"]].head(10).to_html(max_cols=None)))
            num_nonzero_communities += 1

        i += 1
        
    community_coverage_df = pd.DataFrame(rows)
    
    print(f"{num_nonzero_communities} out of {len(communities)} communities had significant GO terms.")
    return important_terms,community_coverage_df

In [29]:
term = "Nuclear Pore Organization (GO:0006999)"
print(get_go_ancestors_at_depth(get_goid(term), depth=1, include_relations=("is_a", "part_of")))

{'GO:0009987'}


In [30]:
go_important_terms, go_community_coverage = enrichment(COMMUNITIES_HGNC,
                                                       TERM_SCORE_CAP,
                                                       PERCENTAGE,
                                                       ['GO_Biological_Process_2023',
                                                        'GO_Molecular_Function_2023',
                                                        'GO_Cellular_Component_2023'],
                                                       lambda term: [go[id].name for id in list(get_go_ancestors_at_depth(get_goid(term), depth=1, include_relations=("is_a", "part_of")))])

Size of community: 909
Number of filtered terms: 8
Number of unmapped terms: 0


C:\Users\celem\AppData\Local\Temp\ipykernel_42364\1924290263.py:61: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  important_terms = pd.concat([important_terms, filtered], ignore_index=True)


,Community Index,Term,Overlap,Adjusted P-value,Category
0,0,Intracellular Protein Transport (GO:0006886),44/325,8.978710e-08,"[localization, cellular process]"
1,0,Protein Transport (GO:0015031),43/313,8.978710e-08,[localization]
2,0,Protein Localization (GO:0008104),45/351,2.200341e-07,[localization]
2301,0,trans-Golgi Network (GO:0005802),35/241,2.735581e-07,[cellular anatomical structure]
3,0,Golgi Vesicle Transport (GO:0048193),31/197,6.947710e-07,"[localization, cellular process]"
4,0,Vesicle-Mediated Transport (GO:0016192),47/411,2.349323e-06,"[localization, cellular process]"
5,0,Endoplasmic Reticulum To Golgi Vesicle-Mediated Transport (GO:0006888),22/115,2.923933e-06,"[localization, cellular process]"
2302,0,COPII-coated ER To Golgi Transport Vesicle (GO:0030134),17/76,4.572532e-06,[cellular anatomical structure]


Size of community: 985
Number of filtered terms: 16
Number of unmapped terms: 1


,Community Index,Term,Overlap,Adjusted P-value,Category
2778,1,Collagen-Containing Extracellular Matrix (GO:0062023),103/373,1.474170e-46,[]
0,1,Extracellular Matrix Organization (GO:0030198),47/176,9.895660e-19,[cellular process]
1,1,Supramolecular Fiber Organization (GO:0097435),51/316,7.040862e-11,[cellular process]
2779,1,Cell-Substrate Junction (GO:0030055),54/395,1.002820e-09,[cellular anatomical structure]
2780,1,Focal Adhesion (GO:0005925),53/387,1.002820e-09,[cellular anatomical structure]
2781,1,Endoplasmic Reticulum Lumen (GO:0005788),43/284,2.539456e-09,[cellular anatomical structure]
2782,1,Basement Membrane (GO:0005604),16/46,1.011990e-08,[cellular anatomical structure]
2783,1,Cytoskeleton (GO:0005856),67/599,1.011990e-08,[cellular anatomical structure]
2784,1,Cell-Cell Junction (GO:0005911),42/299,2.618780e-08,[cellular anatomical structure]
2785,1,Actin Cytoskeleton (GO:0015629),44/327,3.612803e-08,[cellular anatomical structure]


...

Size of community: 300
Number of filtered terms: 5
Number of unmapped terms: 0


,Community Index,Term,Overlap,Adjusted P-value,Category
21,10,Olfactory Receptor Activity (GO:0004984),250/362,0.000000e+00,[molecular transducer activity]
0,10,Sensory Perception Of Smell (GO:0007608),150/230,1.251873e-229,[multicellular organismal process]
1,10,Detection Of Chemical Stimulus Involved In Sensory Perception (GO:0050907),91/141,8.270974e-134,[response to stimulus]
2,10,Detection Of Chemical Stimulus Involved In Sensory Perception Of Smell (GO:0050911),90/139,1.215873e-132,[response to stimulus]
3,10,Sensory Perception Of Chemical Stimulus (GO:0007606),67/110,5.372131e-95,[multicellular organismal process]


Size of community: 134
Number of filtered terms: 23
Number of unmapped terms: 2


,Community Index,Term,Overlap,Adjusted P-value,Category
1473,11,Histone Deacetylase Activity (GO:0004407),9/18,2.133268e-13,[catalytic activity]
0,11,Protein Deacetylation (GO:0006476),10/40,1.364160e-10,[cellular process]
1475,11,Serotonin Receptor Activity (GO:0099589),8/26,3.448323e-10,[molecular transducer activity]
1474,11,"Hydrolase Activity, Acting On Carbon-Nitrogen (But Not Peptide) Bonds, In Linear Amides (GO:0016811)",10/56,3.448323e-10,[catalytic activity]
1,11,"G Protein-Coupled Receptor Signaling Pathway, Coupled To Cyclic Nucleotide Second Messenger (GO:0007187)",10/50,7.811799e-10,"[biological regulation, cellular process]"
1478,11,G Protein-Coupled Neurotransmitter Receptor Activity (GO:0099528),5/6,2.777558e-09,[molecular transducer activity]
1477,11,G Protein-Coupled Acetylcholine Receptor Activity (GO:0016907),5/6,2.777558e-09,[molecular transducer activity]
1476,11,G Protein-Coupled Serotonin Receptor Activity (GO:0004993),7/21,2.777558e-09,[molecular transducer activity]
1479,11,G Protein-Coupled Amine Receptor Activity (GO:0008227),7/28,1.735636e-08,[molecular transducer activity]
2,11,Adenylate Cyclase-Inhibiting G Protein-Coupled Receptor Signaling Pathway (GO:0007193),9/52,2.502757e-08,"[biological regulation, cellular process]"


11 out of 12 communities had significant GO terms.


In [31]:
go_important_terms

,Community Index,Community Size,Term,Overlap,Adjusted P-value,Category,Gene_set,P-value,Old P-value,Old Adjusted P-value,Odds Ratio,Combined Score,Genes,Overlap (value)
0,0,909,Intracellular Protein Transport (GO:0006886),44/325,8.978710e-08,"[localization, cellular process]",GO_Biological_Process_2023,9.214273e-11,0.0,0.0,3.405015,78.682008,ARF3;RTN2;ARF4;STX12;C17ORF75;C2CD5;TMED10;PPP...,0.135385
1,0,909,Protein Transport (GO:0015031),43/313,8.978710e-08,[localization],GO_Biological_Process_2023,9.391956e-11,0.0,0.0,3.461222,79.914717,ARF3;ARF4;STX12;C17ORF75;TMED10;STX16;RAB3D;MI...,0.137380
2,0,909,Protein Localization (GO:0008104),45/351,2.200341e-07,[localization],GO_Biological_Process_2023,3.452417e-10,0.0,0.0,3.197338,69.659687,CUTA;ARF3;ARF4;STX12;C17ORF75;TMED10;STX16;WDR...,0.128205
3,0,909,trans-Golgi Network (GO:0005802),35/241,2.735581e-07,[cellular anatomical structure],GO_Cellular_Component_2023,1.184234e-09,0.0,0.0,3.671186,75.458178,SCOC;C17ORF75;STX16;ATP2C1;BICD1;RABEPK;ECPAS;...,0.145228
4,0,909,Golgi Vesicle Transport (GO:0048193),31/197,6.947710e-07,"[localization, cellular process]",GO_Biological_Process_2023,1.453496e-09,0.0,0.0,4.025270,81.911397,ARF3;ARF4;TMED10;PITPNB;GOSR2;GOSR1;MIA2;TEX26...,0.157360
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
834,11,134,Xenobiotic Catabolic Process (GO:0042178),6/19,2.312521e-07,[cellular process],GO_Biological_Process_2023,2.040921e-09,0.0,0.0,71.585337,1432.412894,GSTM3;CYP2C8;GSTM1;CYP2B6;TPMT;CYP3A5,0.315789
835,11,134,Protein Deacylation (GO:0035601),6/20,3.050832e-07,[cellular process],GO_Biological_Process_2023,2.899637e-09,0.0,0.0,66.468750,1306.687913,HDAC4;HDAC3;HDAC10;HDAC1;HDAC9;HDAC6,0.300000
836,11,134,Negative Regulation Of Myotube Differentiation...,5/11,5.501070e-07,[biological regulation],GO_Biological_Process_2023,5.601905e-09,0.0,0.0,128.294574,2437.617319,HDAC4;HDAC5;XBP1;HDAC3;HDAC1,0.454545
837,11,134,Regulation Of Myotube Differentiation (GO:0010...,6/23,6.838835e-07,[biological regulation],GO_Biological_Process_2023,7.428469e-09,0.0,0.0,54.730699,1024.446260,HDAC4;HDAC5;XBP1;HDAC3;HDAC1;HDAC9,0.260870


In [32]:
go_community_coverage

,community,n_genes,genes_involved,n_involved,n_not_involved
0,0,909,"[AKTIP, AP1G2, AP3D1, AP3M2, AP3S1, AP3S2, AP4...",125,784
1,1,985,"[ABI3BP, ABLIM1, ADAM19, ADAMTS1, ADAMTS2, ADA...",315,670
2,2,1205,"[ACE2, ACKR4, ACOD1, ADAM8, ADAR, ADGRE5, ADGR...",542,663
3,3,907,"[AARS1, AATF, ABCE1, ABT1, ACTR6, AIMP1, ALDH1...",566,341
4,4,931,"[ABHD17A, ABHD17B, ABI1, ABI2, ABLIM2, ABLIM3,...",785,146
5,5,871,"[AASS, ABCA2, ABCA5, ABCA6, ABCA7, ABCA8, ABCA...",253,618
6,6,661,"[ABRAXAS2, ACTL6A, AEBP2, AGO1, AGO2, AIFM2, A...",526,135
7,7,546,"[ABCC8, ADGRL3, ADORA1, ADORA2A, ALX1, ALX3, A...",271,275
8,8,394,"[ANLN, ANP32E, ASF1A, ASF1B, ATAD5, AURKA, AUR...",226,168
9,9,340,[],0,340


### KEGG

In [33]:
# KEGG
def kegg_enrichment(communities,
                    term_score_cap,
                    percentage):
    important_terms = pd.DataFrame(columns=["Community Index","Community Size","Term", "Overlap", "Adjusted P-value","Category"])
    i = 0
    num_nonzero_communities = 0
    for community in communities:
        enr_path = gp.enrichr(
            gene_list=community,
            gene_sets=['KEGG_2021_Human'],
            organism='Human',
            outdir=None
        )
        KEGG_df = enr_path.results

        # Filter by overlap percentage and adjusted p-value
        mask =  (KEGG_df["Adjusted P-value"] < term_score_cap) & (KEGG_df["Overlap"].apply(lambda x: int(x.split("/")[0])/int(x.split("/")[1]) > percentage))
        filtered = KEGG_df[mask].copy()
        
        # Categorization from KEGG Level 2
        filtered["KEGG_ID"] = filtered["Term"].str.replace(r"\s*-\s*Homo sapiens.*$", "", regex=True).str.lower().map(name_to_id)
        filtered["Category"] = filtered["KEGG_ID"].map(get_kegg_level2)
        
        # Sort
        filtered['Overlap (value)'] = filtered['Overlap'].apply(lambda x: int(x.split("/")[0])/int(x.split("/")[1]))
        filtered = filtered.sort_values(['Overlap (value)'], ascending=False)
        
        # Add results to important terms
        if not filtered.empty:
            # print size of community
            print(f"Size of community: {len(community)}")   
            
            # print number of filtered terms
            print(f"Number of filtered terms: {len(filtered)}")
            filtered.loc[:, "Community Index"] = i
            filtered.loc[:, "Community Size"] = len(community)
            important_terms = pd.concat([important_terms, filtered], ignore_index=True)
            
            # show results
            display(HTML(filtered[["Community Index",'Term','Overlap','Adjusted P-value',"KEGG_ID","Category"]].head(10).to_html(max_cols=None)))
            num_nonzero_communities += 1

        i += 1
    print(f"{num_nonzero_communities} out of {len(communities)} communities had significant GO terms.")
    return important_terms

In [34]:
kegg_important_terms, kegg_community_coverage = enrichment(communities = COMMUNITIES_HGNC,
                                 term_score_cap = TERM_SCORE_CAP,
                                 percentage = PERCENTAGE,
                                 db = ['KEGG_2021_Human'],
                                 term_to_category = lambda term: get_kegg_level2(name_to_id.get(term.lower())))

Size of community: 1205
Number of filtered terms: 13
Number of unmapped terms: 0


C:\Users\celem\AppData\Local\Temp\ipykernel_42364\1924290263.py:61: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  important_terms = pd.concat([important_terms, filtered], ignore_index=True)


,Community Index,Term,Overlap,Adjusted P-value,Category
0,2,Cytokine-cytokine receptor interaction,141/295,2.979646e-90,[Signaling molecules and interaction]
1,2,Viral protein interaction with cytokine and cytokine receptor,61/100,8.610662e-47,[Signaling molecules and interaction]
2,2,Hematopoietic cell lineage,35/99,1.450688e-16,[Immune system]
3,2,NF-kappa B signaling pathway,33/104,4.113457e-14,[Signal transduction]
4,2,Chemokine signaling pathway,44/192,3.680852e-13,[Immune system]
5,2,IL-17 signaling pathway,30/94,4.729253e-13,[Immune system]
6,2,JAK-STAT signaling pathway,37/162,3.974348e-11,[Signal transduction]
7,2,Rheumatoid arthritis,26/93,6.058191e-10,[Immune disease]
8,2,Natural killer cell mediated cytotoxicity,31/131,7.695581e-10,[Immune system]
9,2,TNF signaling pathway,28/112,1.543642e-09,[Signal transduction]


Size of community: 907
Number of filtered terms: 5
Number of unmapped terms: 1


,Community Index,Term,Overlap,Adjusted P-value,Category
0,3,Spliceosome,81/150,1.054618e-66,[Transcription]
1,3,Ribosome biogenesis in eukaryotes,41/108,7.615925e-26,[Translation]
2,3,RNA transport,43/186,1.079078e-17,[]
3,3,mRNA surveillance pathway,31/98,4.358251e-17,[Translation]
4,3,Ribosome,33/158,1.947802e-12,[Translation]


...

Size of community: 300
Number of filtered terms: 1
Number of unmapped terms: 0


,Community Index,Term,Overlap,Adjusted P-value,Category
0,10,Olfactory transduction,297/440,0.0,[Sensory system]


Size of community: 134
Number of filtered terms: 3
Number of unmapped terms: 0


,Community Index,Term,Overlap,Adjusted P-value,Category
3,11,Cholinergic synapse,12/113,7.416026e-10,[Nervous system]
5,11,Type I diabetes mellitus,8/43,1.402626e-08,[Endocrine and metabolic disease]
11,11,Graft-versus-host disease,6/42,6.231355e-06,[Immune disease]


9 out of 12 communities had significant GO terms.


In [35]:
kegg_important_terms

,Community Index,Community Size,Term,Overlap,Adjusted P-value,Category,Gene_set,P-value,Old P-value,Old Adjusted P-value,Odds Ratio,Combined Score,Genes,Overlap (value)
0,2,1205,Cytokine-cytokine receptor interaction,141/295,2.979646e-90,[Signaling molecules and interaction],KEGG_2021_Human,1.418879e-92,0.0,0.0,16.040798,3392.435673,CNTFR;IL1RN;CSF3;TNFRSF6B;CSF2;CSF3R;CSF1;CXCL...,0.477966
1,2,1205,Viral protein interaction with cytokine and cy...,61/100,8.610662e-47,[Signaling molecules and interaction],KEGG_2021_Human,8.200630e-49,0.0,0.0,25.643626,2839.325272,CXCL6;CXCL9;CXCL8;IL20;CSF1;IL24;CXCL1;CXCL13;...,0.610000
2,2,1205,Hematopoietic cell lineage,35/99,1.450688e-16,[Immune system],KEGG_2021_Human,2.072412e-18,0.0,0.0,8.755142,356.490265,CSF3;GYPA;CSF2;CSF3R;ITGAM;CSF1;CD1E;CD1D;CD1C...,0.353535
3,2,1205,NF-kappa B signaling pathway,33/104,4.113457e-14,[Signal transduction],KEGG_2021_Human,7.835156e-16,0.0,0.0,7.425516,258.279782,CCL13;CXCL8;EDA;BCL2A1;TNFAIP3;CXCL1;TNFRSF13C...,0.317308
4,2,1205,Chemokine signaling pathway,44/192,3.680852e-13,[Immune system],KEGG_2021_Human,8.763934e-15,0.0,0.0,4.774938,154.555813,CCL14;CX3CR1;CXCL6;CCL13;CXCL9;CXCL8;CCL11;CXC...,0.229167
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
144,8,394,Oocyte meiosis,14/129,4.745840e-06,[Cell growth and death],KEGG_2021_Human,2.636578e-07,0.0,0.0,6.244256,94.591827,SMC3;PKMYT1;SMC1A;AURKA;CDC20;SGO1;CCNB2;PTTG1...,0.108527
145,10,300,Olfactory transduction,297/440,0.000000e+00,[Sensory system],KEGG_2021_Human,0.000000e+00,0.0,0.0,13539.461538,inf,OR7G1;OR8I2;OR9K2;OR2M7;OR11H4;OR2M5;OR52N1;OR...,0.675000
146,11,134,Cholinergic synapse,12/113,7.416026e-10,[Nervous system],KEGG_2021_Human,1.379726e-11,0.0,0.0,19.248499,481.338568,CHRM2;CHRM3;CHRM1;CHRM4;CHRM5;CHAT;BCL2;GNB3;C...,0.106195
147,11,134,Type I diabetes mellitus,8/43,1.402626e-08,[Endocrine and metabolic disease],KEGG_2021_Human,3.914306e-10,0.0,0.0,35.974603,779.253536,IL1A;HLA-DRB5;PTPRN2;HLA-DPB1;LTA;HLA-A;HLA-DR...,0.186047


In [36]:
kegg_community_coverage

,community,n_genes,genes_involved,n_involved,n_not_involved
0,0,909,[],0,909
1,1,985,[],0,985
2,2,1205,"[ACKR4, AIM2, ANPEP, ANTXR1, ANTXR2, B2M, BCL2...",272,933
3,3,907,"[BCAS2, BMS1, BUD31, CASC3, CDC40, CDC5L, CHER...",198,709
4,4,931,"[ABI1, ABI2, ABLIM2, ABLIM3, ACAP1, ACTA2, ACT...",627,304
5,5,871,"[ABCA2, ABCA5, ABCA6, ABCA7, ABCA8, ABCA9, ABC...",148,723
6,6,661,"[ACTL6A, AIFM2, ARID1A, ARID1B, ARID2, ATXN2L,...",276,385
7,7,546,"[ADORA1, ADORA2A, AVPR1A, AVPR1B, BRS3, CALCR,...",62,484
8,8,394,"[AURKA, BUB1, BUB1B, BUB3, CCNA2, CCNB2, CDC20...",39,355
9,9,340,[],0,340


### Reactome

In [37]:
# Reactome enrichment
def reactome_enrichment(communities,
                        term_score_cap,
                        percentage):
    important_terms = pd.DataFrame(columns=["Community Index","Community Size","Term", "Overlap", "Adjusted P-value","Category"])
    i = 0
    num_nonzero_communities = 0
    for community in communities:
        enr_path = gp.enrichr(
            gene_list=community,
            gene_sets=['Reactome_2022'],
            organism='Human',
            outdir=None
        )
        Reactome_df = enr_path.results

        # Filter by overlap percentage and adjusted p-value
        mask =  (Reactome_df["Adjusted P-value"] < term_score_cap) & (Reactome_df["Overlap"].apply(lambda x: int(x.split("/")[0])/int(x.split("/")[1]) > percentage))
        filtered = Reactome_df[mask].copy()
        
        # Categorization from Reactome Level 1
        filtered["Category"] = filtered["Term"].str.extract(r"(R-[A-Z]+-\d+)", expand=False).map(reactome_level1)
        
        # Sort
        filtered['Overlap (value)'] = filtered['Overlap'].apply(lambda x: int(x.split("/")[0])/int(x.split("/")[1]))
        filtered = filtered.sort_values(['Overlap (value)'], ascending=False)
        
        # Add results to important terms
        if not filtered.empty:
            print(f"Size of community: {len(community)}")
            print(f"Number of filtered terms: {len(filtered)}")
            filtered.loc[:, "Community Index"] = i
            filtered.loc[:, "Community Size"] = len(community)
            important_terms = pd.concat([important_terms, filtered], ignore_index=True)
            display(HTML(filtered[["Community Index",'Term','Overlap','Adjusted P-value',"Category"]].head(30).to_html(max_cols=None)))
            num_nonzero_communities += 1
        i += 1
    print(f"{num_nonzero_communities} out of {len(communities)} communities had significant GO terms.")
    return important_terms

In [38]:
reactome_important_terms, reactome_community_coverage = enrichment(COMMUNITIES_HGNC,
                                      TERM_SCORE_CAP,
                                      PERCENTAGE,
                                      ['Reactome_2022'],
                                      lambda term: reactome_level1.get(term.split(" ")[-1],[]))

Size of community: 909
Number of filtered terms: 4
Number of unmapped terms: 0


C:\Users\celem\AppData\Local\Temp\ipykernel_42364\1924290263.py:61: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  important_terms = pd.concat([important_terms, filtered], ignore_index=True)


,Community Index,Term,Overlap,Adjusted P-value,Category
0,0,Intra-Golgi And Retrograde Golgi-to-ER Traffic R-HSA-6811442,33/181,2.333348e-09,[Vesicle-mediated transport]
1,0,Retrograde Transport At Trans-Golgi-Network R-HSA-6811440,16/48,1.962843e-08,[Vesicle-mediated transport]
2,0,Membrane Trafficking R-HSA-199991,64/599,1.962843e-08,[Vesicle-mediated transport]
3,0,Vesicle-mediated Transport R-HSA-5653656,64/637,1.796664e-07,[Vesicle-mediated transport]


Size of community: 985
Number of filtered terms: 10
Number of unmapped terms: 0


,Community Index,Term,Overlap,Adjusted P-value,Category
0,1,Extracellular Matrix Organization R-HSA-1474244,74/291,5.708568e-30,[Extracellular matrix organization]
1,1,Collagen Formation R-HSA-1474290,29/90,3.437667e-14,[Extracellular matrix organization]
2,1,Collagen Biosynthesis And Modifying Enzymes R-HSA-1650814,21/67,6.479693e-10,[Extracellular matrix organization]
3,1,Elastic Fibre Formation R-HSA-1566948,16/39,1.588225e-09,[Extracellular matrix organization]
4,1,Regulation Of IGF Transport And Uptake By IGFBPs R-HSA-381426,27/123,3.896854e-09,[Metabolism of proteins]
5,1,Post-translational Protein Phosphorylation R-HSA-8957275,24/106,1.976000e-08,[Metabolism of proteins]
6,1,Assembly Of Collagen Fibrils And Other Multimeric Structures R-HSA-2022090,16/57,6.033471e-07,[Extracellular matrix organization]
7,1,Platelet Degranulation R-HSA-114608,23/125,2.352458e-06,[Hemostasis]
8,1,Crosslinking Of Collagen Fibrils R-HSA-2243919,7/10,3.720384e-06,[Extracellular matrix organization]
9,1,Response To Elevated Platelet Cytosolic Ca2+ R-HSA-76005,23/130,4.006810e-06,[Hemostasis]


...

Size of community: 300
Number of filtered terms: 3
Number of unmapped terms: 0


,Community Index,Term,Overlap,Adjusted P-value,Category
0,10,Sensory Perception R-HSA-9709957,292/616,0.0,[Sensory Perception]
1,10,Olfactory Signaling Pathway R-HSA-381753,292/401,0.0,[Sensory Perception]
2,10,Expression And Translocation Of Olfactory Receptors R-HSA-9752946,292/393,0.0,[Sensory Perception]


Size of community: 134
Number of filtered terms: 10
Number of unmapped terms: 0


,Community Index,Term,Overlap,Adjusted P-value,Category
0,11,Amine Ligand-Binding Receptors R-HSA-375280,14/40,2.144077e-18,[Signal Transduction]
2,11,Notch-HLH Transcription Pathway R-HSA-350054,11/28,3.058670e-15,[Gene expression (Transcription)]
7,11,NOTCH1 Intracellular Domain Regulates Transcription R-HSA-2122947,11/48,1.078010e-12,[Signal Transduction]
8,11,Constitutive Signaling By NOTCH1 HD+PEST Domain Mutants R-HSA-2894862,11/58,9.125546e-12,[Disease]
9,11,Signaling By NOTCH1 R-HSA-1980143,11/74,1.377291e-10,[Signal Transduction]
10,11,Muscarinic Acetylcholine Receptors R-HSA-390648,5/5,6.670934e-10,[Signal Transduction]
14,11,Adrenoceptors R-HSA-390696,5/9,6.033097e-08,[Signal Transduction]
20,11,Regulation Of Insulin Secretion R-HSA-422356,8/77,1.309870e-06,[Metabolism]
21,11,RUNX2 Regulates Bone Development R-HSA-8941326,6/31,1.381178e-06,[Gene expression (Transcription)]
24,11,Regulation Of PTEN Gene Transcription R-HSA-8943724,7/61,3.916508e-06,[Signal Transduction]


11 out of 12 communities had significant GO terms.


In [39]:
reactome_important_terms

,Community Index,Community Size,Term,Overlap,Adjusted P-value,Category,Gene_set,P-value,Old P-value,Old Adjusted P-value,Odds Ratio,Combined Score,Genes,Overlap (value)
0,0,909,Intra-Golgi And Retrograde Golgi-to-ER Traffic...,33/181,2.333348e-09,[Vesicle-mediated transport],Reactome_2022,7.431044e-12,0.0,0.0,4.821663,1.235568e+02,ARF3;ARF4;SCOC;TMF1;STX16;GOSR2;GOSR1;GOLIM4;G...,0.182320
1,0,909,Retrograde Transport At Trans-Golgi-Network R-...,16/48,1.962843e-08,[Vesicle-mediated transport],Reactome_2022,1.668892e-10,0.0,0.0,10.671333,2.402511e+02,SCOC;NAA30;TMF1;STX16;M6PR;GCC1;RABEPK;TGOLN2;...,0.333333
2,0,909,Membrane Trafficking R-HSA-199991,64/599,1.962843e-08,[Vesicle-mediated transport],Reactome_2022,1.875328e-10,0.0,0.0,2.626962,5.883625e+01,ARF3;ARF4;SCOC;MCFD2;C2CD5;GOLIM4;GCC1;CLINT1;...,0.106845
3,0,909,Vesicle-mediated Transport R-HSA-5653656,64/637,1.796664e-07,[Vesicle-mediated transport],Reactome_2022,2.288744e-09,0.0,0.0,2.447726,4.869814e+01,ARF3;ARF4;SCOC;MCFD2;C2CD5;GOLIM4;GCC1;CLINT1;...,0.100471
4,1,985,Extracellular Matrix Organization R-HSA-1474244,74/291,5.708568e-30,[Extracellular matrix organization],Reactome_2022,1.235621e-32,0.0,0.0,7.036639,5.169899e+02,DDR1;COL18A1;SPARC;COL14A1;COL12A1;LOXL3;PLOD2...,0.254296
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
650,11,134,Muscarinic Acetylcholine Receptors R-HSA-390648,5/5,6.670934e-10,[Signal Transduction],Reactome_2022,1.252223e-11,0.0,0.0,99330.000000,2.493532e+06,CHRM2;CHRM3;CHRM1;CHRM4;CHRM5,1.000000
651,11,134,Adrenoceptors R-HSA-390696,5/9,6.033097e-08,[Signal Transduction],Reactome_2022,1.544308e-09,0.0,0.0,192.461240,3.904786e+03,ADRB3;ADRB2;ADRA2C;ADRA2B;ADRA2A,0.555556
652,11,134,Regulation Of Insulin Secretion R-HSA-422356,8/77,1.309870e-06,[Metabolism],Reactome_2022,4.694075e-08,0.0,0.0,18.216701,3.073955e+02,GLP1R;CHRM3;GNB3;CACNA1D;GCG;CACNA1C;ADRA2C;AD...,0.103896
653,11,134,RUNX2 Regulates Bone Development R-HSA-8941326,6/31,1.381178e-06,[Gene expression (Transcription)],Reactome_2022,5.185309e-08,0.0,0.0,37.201875,6.240559e+02,HDAC4;COL1A1;HDAC3;SRC;BGLAP;HDAC6,0.193548


In [40]:
reactome_community_coverage

,community,n_genes,genes_involved,n_involved,n_not_involved
0,0,909,"[ACBD3, AP1G2, AP3S1, AP4S1, ARF3, ARF4, ARFRP...",64,845
1,1,985,"[ADAM12, ADAM19, ADAMTS1, ADAMTS2, ADAMTS5, AG...",117,868
2,2,1205,"[ACKR4, ADA2, ADAM8, ADAR, ADGRE1, ADGRE2, ADG...",405,800
3,3,907,"[AARS1, AIMP1, AIMP2, AURKAIP1, BCAS2, BMS1, B...",331,576
4,4,931,"[AAMP, ABHD17A, ABHD17B, ABI1, ABI2, ABLIM2, A...",750,181
5,5,871,"[AASS, ABCA2, ABCA5, ABCA6, ABCA7, ABCA8, ABCA...",295,576
6,6,661,"[ABRAXAS2, ACTL6A, AEBP2, AGO1, AGO2, AIFM2, A...",540,121
7,7,546,"[ABCC8, ADORA1, ADORA2A, AVPR1A, AVPR1B, BRS3,...",90,456
8,8,394,"[AURKA, AURKB, BARD1, BORA, BUB1, BUB1B, BUB3,...",148,246
9,9,340,[],0,340


# Important Terms df

In [41]:
community_coverage_combined = go_community_coverage.copy()

community_coverage_combined["genes_involved"] = [
    set(a) | set(b) | set(c)
    for a, b, c in zip(go_community_coverage["genes_involved"], kegg_community_coverage["genes_involved"], reactome_community_coverage["genes_involved"])
]
community_coverage_combined["n_involved"] = community_coverage_combined["genes_involved"].apply(len)
community_coverage_combined["n_not_involved"] = community_coverage_combined["n_genes"] - community_coverage_combined["n_involved"]
community_coverage_combined["percentage_involved"] = community_coverage_combined["n_involved"] / community_coverage_combined["n_genes"]

In [42]:
community_coverage_combined

,community,n_genes,genes_involved,n_involved,n_not_involved,percentage_involved
0,0,909,"{CLN3, HIP1R, TOR1B, ARFGEF1, BLOC1S6, DOP1B, ...",144,765,0.158416
1,1,985,"{PLSCR3, SHROOM3, DAB2, LOXL3, BIN3, STC2, CTN...",345,640,0.350254
2,2,1205,"{SAA1, GDF10, CSF2, CD96, KLRC3, CLEC4G, CD38,...",634,571,0.526141
3,3,907,"{TARS1, IMMP1L, HEATR1, AURKAIP1, NOL6, SF3A3,...",583,324,0.642778
4,4,931,"{PDGFRB, GRIA3, PIK3C2G, DENND4A, IRS1, EFNA3,...",887,44,0.952739
5,5,871,"{PC, SLC27A1, DAO, ABCC3, POR, STARD5, GSTA1, ...",360,511,0.413318
6,6,661,"{DNMT3L, CUL2, ZNF195, ZNF430, TNRC6B, ZBTB17,...",629,32,0.951589
7,7,546,"{SHOX, FOXJ1, ASTN1, ECEL1, IRX1, SORCS3, DLX1...",281,265,0.514652
8,8,394,"{KIF4A, TPX2, MZT1, PLK4, KATNA1, DCTPP1, BORA...",245,149,0.621827
9,9,340,{},0,340,0.000000


In [43]:
comm_to_involved_pct = dict(zip(community_coverage_combined["community"], community_coverage_combined["percentage_involved"]))

with open(DISEASE_FOLDER + "comm_to_involved_pct.json", "w") as f:
    json.dump(comm_to_involved_pct, f, indent=2)


In [44]:
important_terms = pd.DataFrame(columns=["Community Index","Community Size","Term", "Overlap", "Adjusted P-value","Category"])
c = [go_important_terms,kegg_important_terms,reactome_important_terms]
important_terms = pd.concat(c, ignore_index=True)
important_terms = important_terms.sort_values(by="Community Index")
important_terms

,Community Index,Community Size,Term,Overlap,Adjusted P-value,Category,Gene_set,P-value,Old P-value,Old Adjusted P-value,Odds Ratio,Combined Score,Genes,Overlap (value)
0,0,909,Intracellular Protein Transport (GO:0006886),44/325,8.978710e-08,"[localization, cellular process]",GO_Biological_Process_2023,9.214273e-11,0.0,0.0,3.405015,78.682008,ARF3;RTN2;ARF4;STX12;C17ORF75;C2CD5;TMED10;PPP...,0.135385
991,0,909,Vesicle-mediated Transport R-HSA-5653656,64/637,1.796664e-07,[Vesicle-mediated transport],Reactome_2022,2.288744e-09,0.0,0.0,2.447726,48.698143,ARF3;ARF4;SCOC;MCFD2;C2CD5;GOLIM4;GCC1;CLINT1;...,0.100471
989,0,909,Retrograde Transport At Trans-Golgi-Network R-...,16/48,1.962843e-08,[Vesicle-mediated transport],Reactome_2022,1.668892e-10,0.0,0.0,10.671333,240.251086,SCOC;NAA30;TMF1;STX16;M6PR;GCC1;RABEPK;TGOLN2;...,0.333333
988,0,909,Intra-Golgi And Retrograde Golgi-to-ER Traffic...,33/181,2.333348e-09,[Vesicle-mediated transport],Reactome_2022,7.431044e-12,0.0,0.0,4.821663,123.556832,ARF3;ARF4;SCOC;TMF1;STX16;GOSR2;GOSR1;GOLIM4;G...,0.182320
7,0,909,COPII-coated ER To Golgi Transport Vesicle (GO...,17/76,4.572532e-06,[cellular anatomical structure],GO_Cellular_Component_2023,3.958902e-08,0.0,0.0,6.147754,104.786709,MCFD2;SEC23A;TMED10;GOSR2;SEC16A;TEX261;ECPAS;...,0.223684
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
837,11,134,Regulation Of Myotube Differentiation (GO:0010...,6/23,6.838835e-07,[biological regulation],GO_Biological_Process_2023,7.428469e-09,0.0,0.0,54.730699,1024.446260,HDAC4;HDAC5;XBP1;HDAC3;HDAC1;HDAC9,0.260870
838,11,134,Negative Regulation Of Striated Muscle Cell Di...,5/17,4.864172e-06,[biological regulation],GO_Biological_Process_2023,7.264886e-08,0.0,0.0,64.127907,1054.110684,HDAC4;HDAC5;XBP1;HDAC3;HDAC1,0.294118
985,11,134,Cholinergic synapse,12/113,7.416026e-10,[Nervous system],KEGG_2021_Human,1.379726e-11,0.0,0.0,19.248499,481.338568,CHRM2;CHRM3;CHRM1;CHRM4;CHRM5;CHAT;BCL2;GNB3;C...,0.106195
828,11,134,Adenylate Cyclase-Inhibiting G Protein-Coupled...,5/7,6.387042e-08,"[biological regulation, cellular process]",GO_Biological_Process_2023,2.601647e-10,0.0,0.0,384.961240,8495.981539,CHRM2;CHRM3;CHRM4;CHRM5;HRH4,0.714286


## Remove Redundant Terms

In [45]:
important_terms

,Community Index,Community Size,Term,Overlap,Adjusted P-value,Category,Gene_set,P-value,Old P-value,Old Adjusted P-value,Odds Ratio,Combined Score,Genes,Overlap (value)
0,0,909,Intracellular Protein Transport (GO:0006886),44/325,8.978710e-08,"[localization, cellular process]",GO_Biological_Process_2023,9.214273e-11,0.0,0.0,3.405015,78.682008,ARF3;RTN2;ARF4;STX12;C17ORF75;C2CD5;TMED10;PPP...,0.135385
991,0,909,Vesicle-mediated Transport R-HSA-5653656,64/637,1.796664e-07,[Vesicle-mediated transport],Reactome_2022,2.288744e-09,0.0,0.0,2.447726,48.698143,ARF3;ARF4;SCOC;MCFD2;C2CD5;GOLIM4;GCC1;CLINT1;...,0.100471
989,0,909,Retrograde Transport At Trans-Golgi-Network R-...,16/48,1.962843e-08,[Vesicle-mediated transport],Reactome_2022,1.668892e-10,0.0,0.0,10.671333,240.251086,SCOC;NAA30;TMF1;STX16;M6PR;GCC1;RABEPK;TGOLN2;...,0.333333
988,0,909,Intra-Golgi And Retrograde Golgi-to-ER Traffic...,33/181,2.333348e-09,[Vesicle-mediated transport],Reactome_2022,7.431044e-12,0.0,0.0,4.821663,123.556832,ARF3;ARF4;SCOC;TMF1;STX16;GOSR2;GOSR1;GOLIM4;G...,0.182320
7,0,909,COPII-coated ER To Golgi Transport Vesicle (GO...,17/76,4.572532e-06,[cellular anatomical structure],GO_Cellular_Component_2023,3.958902e-08,0.0,0.0,6.147754,104.786709,MCFD2;SEC23A;TMED10;GOSR2;SEC16A;TEX261;ECPAS;...,0.223684
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
837,11,134,Regulation Of Myotube Differentiation (GO:0010...,6/23,6.838835e-07,[biological regulation],GO_Biological_Process_2023,7.428469e-09,0.0,0.0,54.730699,1024.446260,HDAC4;HDAC5;XBP1;HDAC3;HDAC1;HDAC9,0.260870
838,11,134,Negative Regulation Of Striated Muscle Cell Di...,5/17,4.864172e-06,[biological regulation],GO_Biological_Process_2023,7.264886e-08,0.0,0.0,64.127907,1054.110684,HDAC4;HDAC5;XBP1;HDAC3;HDAC1,0.294118
985,11,134,Cholinergic synapse,12/113,7.416026e-10,[Nervous system],KEGG_2021_Human,1.379726e-11,0.0,0.0,19.248499,481.338568,CHRM2;CHRM3;CHRM1;CHRM4;CHRM5;CHAT;BCL2;GNB3;C...,0.106195
828,11,134,Adenylate Cyclase-Inhibiting G Protein-Coupled...,5/7,6.387042e-08,"[biological regulation, cellular process]",GO_Biological_Process_2023,2.601647e-10,0.0,0.0,384.961240,8495.981539,CHRM2;CHRM3;CHRM4;CHRM5;HRH4,0.714286


In [46]:
TERM_SIZE_CAP = 100

In [47]:
def parse_genes(gene_str):
    # Enrichr "Genes" field is like "IL6;STAT3;JAK2"
    return set(re.split(r"[;, ]+", gene_str.strip()))

df = important_terms.sort_values("Adjusted P-value")

kept_rows = []
kept_gene_sets = []

for _, row in df.iterrows():
    genes = parse_genes(row["Genes"])
    if len(genes) == 0:
        continue
    too_similar = any(len(genes & g)/len(genes | g) > 0.35 for g in kept_gene_sets)
    if not too_similar and int(row['Overlap'].split("/")[1]) < TERM_SIZE_CAP:
        kept_rows.append(row)
        kept_gene_sets.append(genes)

important_terms_nonredundant = pd.DataFrame(kept_rows).sort_values(by=["Community Index", "Adjusted P-value"])

In [48]:
important_terms_nonredundant

,Community Index,Community Size,Term,Overlap,Adjusted P-value,Category,Gene_set,P-value,Old P-value,Old Adjusted P-value,Odds Ratio,Combined Score,Genes,Overlap (value)
989,0,909,Retrograde Transport At Trans-Golgi-Network R-...,16/48,1.962843e-08,[Vesicle-mediated transport],Reactome_2022,1.668892e-10,0.0,0.0,10.671333,240.251086,SCOC;NAA30;TMF1;STX16;M6PR;GCC1;RABEPK;TGOLN2;...,0.333333
7,0,909,COPII-coated ER To Golgi Transport Vesicle (GO...,17/76,4.572532e-06,[cellular anatomical structure],GO_Cellular_Component_2023,3.958902e-08,0.0,0.0,6.147754,104.786709,MCFD2;SEC23A;TMED10;GOSR2;SEC16A;TEX261;ECPAS;...,0.223684
993,1,985,Collagen Formation R-HSA-1474290,29/90,3.437667e-14,[Extracellular matrix organization],Reactome_2022,1.488168e-16,0.0,0.0,9.425646,343.506525,COL18A1;COL15A1;COL13A1;PCOLCE2;COL14A1;COL11A...,0.322222
995,1,985,Elastic Fibre Formation R-HSA-1566948,16/39,1.588225e-09,[Extracellular matrix organization],Reactome_2022,1.375086e-11,0.0,0.0,13.634495,340.997635,FBN2;LOXL3;FBLN1;LTBP3;FBLN2;FBLN5;LOXL1;LOXL2...,0.410256
14,1,985,Basement Membrane (GO:0005604),16/46,1.011990e-08,[cellular anatomical structure],GO_Cellular_Component_2023,2.610145e-10,0.0,0.0,10.449260,230.578030,COL15A1;DST;TNC;P3H2;NID1;NID2;THBS4;LOXL2;SMO...,0.347826
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
829,11,134,Positive Regulation Of Adenylate Cyclase Activ...,7/26,6.440446e-08,[biological regulation],GO_Biological_Process_2023,3.060633e-10,0.0,0.0,57.575218,1261.313493,GLP1R;ADRB3;GIPR;CACNA1D;ADRB2;CACNA1C;DRD5,0.269231
834,11,134,Xenobiotic Catabolic Process (GO:0042178),6/19,2.312521e-07,[cellular process],GO_Biological_Process_2023,2.040921e-09,0.0,0.0,71.585337,1432.412894,GSTM3;CYP2C8;GSTM1;CYP2B6;TPMT;CYP3A5,0.315789
836,11,134,Negative Regulation Of Myotube Differentiation...,5/11,5.501070e-07,[biological regulation],GO_Biological_Process_2023,5.601905e-09,0.0,0.0,128.294574,2437.617319,HDAC4;HDAC5;XBP1;HDAC3;HDAC1,0.454545
1640,11,134,Regulation Of Insulin Secretion R-HSA-422356,8/77,1.309870e-06,[Metabolism],Reactome_2022,4.694075e-08,0.0,0.0,18.216701,307.395533,GLP1R;CHRM3;GNB3;CACNA1D;GCG;CACNA1C;ADRA2C;AD...,0.103896


## Save Important Terms

In [49]:
important_terms.to_csv(f"../output/{DISEASE}/important_terms.csv", index=False)

# Robustness Analysis

In [50]:
# def run_enrichment_func(community,term_score_cap,percentage):
#     # GO df
#     enr_go = gp.enrichr(
#         gene_list=community,
#         gene_sets=['GO_Biological_Process_2023',
#                 'GO_Molecular_Function_2023',
#                 'GO_Cellular_Component_2023'],
#         organism='Human',
#         outdir=None # don't write to disk
#     )
#     GO_df = enr_go.results
#     mask =  (GO_df["Adjusted P-value"] < term_score_cap) & (GO_df["Overlap"].apply(lambda x: int(x.split("/")[0])/int(x.split("/")[1]) > percentage))
#     GO_df = GO_df[mask].copy()   
    
#     # KEGG df
#     enr_kegg = gp.enrichr(
#         gene_list=community,
#         gene_sets=['KEGG_2021_Human'],
#         organism='Human',
#         outdir=None
#     )
#     KEGG_df = enr_kegg.results
#     mask =  (KEGG_df["Adjusted P-value"] < term_score_cap) & (KEGG_df["Overlap"].apply(lambda x: int(x.split("/")[0])/int(x.split("/")[1]) > percentage))
#     KEGG_df = KEGG_df[mask].copy() 
       
#     # Reactome df
#     enr_reactome = gp.enrichr(
#         gene_list=community,
#         gene_sets=['Reactome_2022'],
#         organism='Human',
#         outdir=None
#     )
#     Reactome_df = enr_reactome.results  
#     mask =  (Reactome_df["Adjusted P-value"] < term_score_cap) & (Reactome_df["Overlap"].apply(lambda x: int(x.split("/")[0])/int(x.split("/")[1]) > percentage))
#     Reactome_df = Reactome_df[mask].copy()
    
    
#     all_df = [GO_df,KEGG_df,Reactome_df]
#     # build result df by concatenating
#     result = pd.concat(all_df, ignore_index=True)
#     return result

In [51]:
# from json import JSONDecodeError

# # ---------------- 1) Safe wrapper that calls YOUR enrichr function ----------------
# _ENR_CACHE = {}  # key: tuple(sorted(genes)) -> DataFrame (copy)

# def run_enrichment_safe(run_enrichment_func, community, retries=5, base_sleep=0.8):
#     """
#     Calls user's run_enrichment_func(community) with retries + memoization.
#     Returns a DataFrame (possibly empty). Never raises JSONDecodeError outward.
#     """
#     # Ensure we always pass a list of gene symbols (never a bare string)
#     genes = np.atleast_1d(np.array(community, dtype=object)).tolist()
#     if len(genes) == 0:
#         return pd.DataFrame()

#     key = tuple(sorted(genes))
#     if key in _ENR_CACHE:
#         return _ENR_CACHE[key].copy()

#     for a in range(retries):
#         try:
#             df = run_enrichment_func(genes,TERM_SCORE_CAP,PERCENTAGE)
#             if df is None:
#                 # treat as transient failure to trigger retry
#                 raise RuntimeError("run_enrichment_func returned None")
#             _ENR_CACHE[key] = df.copy()
#             return df
#         except (JSONDecodeError, OSError, RuntimeError, ValueError) as e:
#             # Transient errors from HTTP/JSON/file handling inside gseapy
#             if a == retries - 1:
#                 # Give up: return empty so pipeline continues
#                 return pd.DataFrame()
#             time.sleep(base_sleep * (2 ** a) + np.random.rand() * 0.3)

#     return pd.DataFrame()

# # ---------------- 2) Minimal bootstrap to record robust terms ----------------
# def get_robust_terms(communities_HGNC, run_enrichment_func,
#                      R=50, leaveout=0.10, recurrence_cutoff=0.70, seed=42):
#     """
#     Uses YOUR run_enrichment_func(community)->DataFrame (already filtered to significant terms).
#     Returns DataFrame with columns: community_id, term, recurrence (and Gene_set if available).
#     """
#     rng = np.random.default_rng(seed)
#     rows = []

#     for cid, community in enumerate(communities_HGNC):
#         n = len(community)
#         if n == 0:
#             continue
#         drop_k = max(1, int(np.floor(leaveout * n)))
#         counts = Counter()

#         for _ in range(R):
#             # Jackknife subset (ensure not empty)
#             keep = np.ones(n, dtype=bool)
#             keep[rng.choice(n, size=min(drop_k, n), replace=False)] = False
#             sub = np.atleast_1d(np.array(community, dtype=object)[keep]).tolist()
#             if len(sub) == 0:
#                 continue

#             df = run_enrichment_safe(run_enrichment_func, sub)
#             if df is None or df.empty:
#                 continue

#             # Your function already returns significant terms; just count them.
#             # If it includes multiple libraries, preserve Gene_set to disambiguate names.
#             if 'Term' not in df.columns:
#                 continue  # be defensive

#             if 'Gene_set' in df.columns:
#                 terms = (df[['Term', 'Gene_set']]
#                          .dropna()
#                          .drop_duplicates()
#                          .apply(lambda r: f"{r['Term']}|{r['Gene_set']}", axis=1)
#                          .tolist())
#             else:
#                 terms = df['Term'].dropna().drop_duplicates().tolist()

#             counts.update(terms)

#             # tiny pause helps with API rate limits if your func calls Enrichr internally
#             time.sleep(0.03)

#         # Keep only robust terms
#         for t, c in counts.items():
#             freq = c / max(R, 1)
#             if freq >= recurrence_cutoff:
#                 if '|' in t:
#                     term, gene_set = t.split('|', 1)
#                     rows.append({'Community Index': cid, 'Term': term, 'recurrence': freq, 'Gene_set': gene_set})
#                 else:
#                     rows.append({'Community Index': cid, 'Term': t, 'recurrence': freq})

#     return (pd.DataFrame(rows)
#               .sort_values(['Community Index', 'recurrence'], ascending=[True, False])
#               .reset_index(drop=True))

In [52]:
# twr3 = get_robust_terms([COMMUNITIES_HGNC[1]], run_enrichment_func,
#                                 R=25, leaveout=0.1, recurrence_cutoff=0)

In [53]:
# twr3

In [54]:
# terms_with_recurrence = get_robust_terms(COMMUNITIES_HGNC, run_enrichment_func,
#                                 R=10, leaveout=0.1, recurrence_cutoff=0)

In [55]:
# terms_with_recurrence

In [56]:
# # rename important terms to match terms_with_recurrence
# important_terms = important_terms.rename(columns={'index': 'community_id'})
# important_terms = important_terms.rename(columns={'Term': 'term'})

In [57]:
# terms_with_rec_merged = important_terms.merge(
#     terms_with_recurrence[['community_id', 'term', 'Gene_set', 'recurrence']],
#     on=['community_id', 'term', 'Gene_set'],
#     how='left'
# )

# terms_with_rec_merged['recurrence'] = terms_with_rec_merged['recurrence'].fillna(0.0)

# terms_with_rec_merged = terms_with_rec_merged.sort_values(
#     ['community_id', 'recurrence'],
#     ascending=[True, False]
# ).reset_index(drop=True)

In [58]:
# terms_with_rec_merged

In [59]:
# community_summary = (
#     terms_with_rec_merged
#     .groupby("community_id")["recurrence"]
#     .agg(mean_recurrence="mean", term_count="count")
#     .reset_index()
# )

# print(community_summary)

In [60]:
# display(HTML(terms_with_recurrence.to_html(max_cols=None)))

# Checks!

In [61]:
DGIDB_genes_ncbi = list(DGIDB_gene_to_index.keys())

In [62]:
def DGIDB_count(c):
    return len(set(c) & set(DGIDB_genes_ncbi))